# Pya Examples AGen Synth Control using the @asynth decorator

This notebook demonstrates how to define synthesizers with control nodes.
- Control nodes allow to change values or replace AGens during rendering.
- This turns AGens, e.g. when played directly on Aserver, a highly interactive live performance engine

In [ ]:
from pya import startup
from pya.gui import AserverGUI
asgui = AserverGUI()
s = startup()

## Explicit definition of a synth with control nodes

Let's create a SinOsc synth with gate, amp, and freq that can be modulated.
- Note that we will have the release time as parameter that should not be modulated.
- The synth is defined via a function (here syn), allowing to set default values
- When nodes are needed (so that we can easily modulate later)
  - they are explicitly created using the Node constructor
- we then pass the nodes to the AGens (which now works after a core AGen patch)
- ultimately we store the nodes under the names of the argument names for later control

After creation of an instance, we can then update values in a Pythonic fashion
- e.g. to a new value
- or, replacing the value by another AGen 
- note that setting the gate to 0 triggers the release phase.

In [ ]:
import time
from pya.agen.core import AGen, ControlDict
from pya.agen.lib import Release, SinOsc
Node = AGen.Node

def syn(freq=200, gate=1, amp=0.1, release=1.0):
    freq_node = Node(freq, True)
    gate_node = Node(gate, True)
    ag = SinOsc(freq_node) * amp * Release(gate_node, release)
    ag.ctrl = ControlDict(dict(freq=freq_node, gate=gate_node))
    return ag

ag1 = syn(200).play()

time.sleep(0.5)
ag1.ctrl.freq = 300

time.sleep(0.5)
ag1.ctrl.freq = SinOsc(20) * 20 + 400  # works with the experimental generate patch

time.sleep(0.5)
ag1.ctrl.gate = 0

Enabling arithmetic operations when needed:

- Note that it is not possible to compute with nodes:
  - `SinOsc(2 * freq_node)` will not work.
- The Gen class is a trivial AGen which only generates samples from its argument.
  - is turns a Node into an AGen, so maybe Node2AGen would be a better name...
  - Disclaimer: the name is subject to change in future versions
- Using Gen we can write `SinOsc(2*Gen(freq_note)) and thus
  - gain the possibility for inner computing
  - while maintaining the possibility to control freq.
- Here is an example

In [ ]:
from pya.agen.core import Gen
from pya.agen.lib import Line

def syn(freq=200):
    freq_node = Node(freq, True)
    ag = SinOsc(2 * Gen(freq_node)) * Line(0.2, 0, 2)
    ag.ctrl = ControlDict(dict(freq=freq_node))
    return ag

ag1 = syn(200).play()

time.sleep(0.5)
ag1.ctrl.freq = SinOsc(20) * 20 + 400  

time.sleep(0.5)
ag1.ctrl.freq = 300


Node has now an agen property which actually creates Gen(self)
- so an alternative to `Gen(freq_node)` which avoids brackets is `freq_node.agen`.

In [ ]:
def syn(freq=200):
    freq_node = Node(freq, True)
    ag = SinOsc(2 * freq_node.agen) * Line(0.2, 0, 0.4)
    ag.ctrl = ControlDict(dict(freq=freq_node))
    return ag

ag1 = syn(200).play()
time.sleep(0.2)
ag1.ctrl.freq = 300


## The @asynth decorator

The asynth decorator wraps the syn functions with
- automatic argument to node conversion before calling the synth
- automatically adding the ctrl attribute in the resulting AGen
- Note that your function has to (and should) return an AGen instance.

A synth thus becomes significantly more elegant. For instance

```python
def syn(freq=200, gate=1, amp=0.1, release=1.0):
    freq_node = Node(freq, True)
    gate_node = Node(gate, True)
    ag = SinOsc(freq_node) * amp * Release(gate_node, release)
    ag.ctrl = ControlDict(dict(freq=freq_node, gate=gate_node))
    return ag
```
will become:

In [ ]:
from pya.agen.core import asynth

@asynth
def syn(freq=200, gate=1, amp=0.1, release: float = 1.0):
    return SinOsc(freq) * amp * Release(gate, release)

ag1 = syn(200).play()
time.sleep(0.2)
ag1.ctrl.freq = 300
time.sleep(0.8)
ag1.ctrl.gate = 0


- Note that the default is to turn arguments into nodes!
- Arguments that are given a typetag are excluded
  - this is important as many synths will need additional arguments that cannot be modulated.
- Disclaimer: 
  - this choice is experimental and may change!
  - At this time even a Node or GenOrNum type tag will let asynth ignore it

**Example: MIDI-synthesizer processing NoteOn and NoteOff messages**

In [ ]:
import pyamapping as pam
from pya.agen.lib import SinOsc, Pan2, Release
from pya.agen.midi import MIDI_ctrl

midiport = 1
mc = MIDI_ctrl(midiport, True) # select Port as needed

@asynth
def syn1(freq=100, amp=0.1, pos=0, gate=1, reldur: float = 0.2):
    ag = (SinOsc(freq=freq) * amp | Pan2.p(pos)) * Release(gate, reldur)
    return ag

mc.note_list = [[0] for _ in range(127)]

def noteon_fn(note=60, vel=64, chn=0, ctx=None):
    ctx.note_list[note] = syn1(freq=pam.midi_to_cps(note), amp=(vel/256)**2).play()

def noteoff_fn(note=60, vel=64, chn=0, ctx=None):
    ag = ctx.note_list[note]
    if ag is not False:
        ag.ctrl.gate = 0
        ctx.note_list[note] = False

mc.set_callback("NoteOn", noteon_fn)
mc.set_callback("NoteOff", noteoff_fn)

In case you have no MIDI device, let's play some notes via code

In [ ]:
import rtmidi

midiout = rtmidi.MidiOut()
available_ports = midiout.get_ports()

if available_ports:
    midiout.open_port(0) # or Name such as 'IAC Driver')

with midiout:
    # Here you send the message to the device
    for note in range(50, 90, 3):
        midiout.send_message([0x90, note, 90]) # note_on
        time.sleep(0.1)
        midiout.send_message([0x80, note, 0]) # note_off 
    time.sleep(0.5)
    # Here you send the stop message